# LeetCode 1261 — Find Elements in a Contaminated Binary Tree bằng Python

Link đề: https://leetcode.com/problems/find-elements-in-a-contaminated-binary-tree/ · Độ khó: **Medium** · Chủ đề: Hash Table, Tree, Depth-First Search, Breadth-First Search, Design, Binary Tree

Skeleton LeetCode (bài **thiết kế lớp**):

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class FindElements:

    def __init__(self, root: TreeNode | None):
        ...

    def find(self, target: int) -> bool:
        ...

# obj = FindElements(root)
# param_1 = obj.find(target)
```

Ghi chú: LeetCode tự cung cấp `TreeNode` và tự gọi chuỗi lệnh. Trong notebook ta **tự dựng bản giả lập**: lớp `TreeNode`, hàm dựng cây từ mảng kiểu LeetCode (`[-1, None, -1]`) và bộ chạy chuỗi lệnh `["FindElements","find",...]`.

## 1. Mục tiêu học

1. Hiểu quy tắc đánh số cây kiểu **heap**: gốc `0`, con trái `2x+1`, con phải `2x+2` — và khôi phục giá trị bằng **DFS/BFS**.
2. Thiết kế lớp có **tiền xử lý một lần** (`__init__`) để mỗi truy vấn (`find`) nhanh: dùng `set` cho `O(1)`.
3. Khám phá cách **không cần bộ nhớ phụ**: đi theo các **bit của `target + 1`** từ gốc xuống, `O(h)` mỗi truy vấn.
4. Tự viết công cụ test bài thiết kế: dựng cây từ mảng level-order, chạy chuỗi lệnh và so với output mong đợi.
5. Tránh lỗi: nhầm công thức con (`2x`, `2x+1`), dùng `bin(target)` thay vì `bin(target+1)`, đọc bit sai chiều, giả định cây đầy đủ.

## 2. Đề bài & ví dụ

Cho một cây nhị phân thoả quy tắc:
- `root.val == 0`;
- nút có giá trị `x`: nếu có con trái thì con trái mang `2x + 1`; nếu có con phải thì con phải mang `2x + 2`.

Cây đã bị **"nhiễm bẩn"**: mọi `val` bị đổi thành `-1`. Cài đặt lớp `FindElements`:
- `FindElements(root)`: nhận cây bị nhiễm bẩn và **khôi phục** nó.
- `find(target)`: trả `True` nếu `target` có trong cây đã khôi phục.

**Ví dụ 1:** `["FindElements","find","find"]`, `[[[-1,null,-1]],[1],[2]]` → `[null,false,true]` (cây chỉ có gốc `0` và con phải `2`).

**Ví dụ 2:** `["FindElements","find","find","find"]`, `[[[-1,-1,-1,-1,-1]],[1],[3],[5]]` → `[null,true,true,false]`.

**Ví dụ 3:** `["FindElements","find","find","find","find"]`, `[[[-1,null,-1,-1,null,-1]],[2],[3],[4],[5]]` → `[null,true,false,false,true]`.

**Ràng buộc:** mọi `TreeNode.val == -1`; chiều cao cây ≤ 20; số nút trong `[1, 10^4]`; tổng số lần gọi `find` trong `[1, 10^4]`; `0 <= target <= 10^6`.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional

class TreeNode:
    """Bản giả lập TreeNode của LeetCode."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(arr):
    """Dựng cây từ mảng level-order kiểu LeetCode (None = không có nút)."""
    if not arr or arr[0] is None:
        return None
    root = TreeNode(arr[0])
    q = deque([root])
    i = 1
    while q and i < len(arr):
        node = q.popleft()
        if i < len(arr) and arr[i] is not None:
            node.left = TreeNode(arr[i]); q.append(node.left)
        i += 1
        if i < len(arr) and arr[i] is not None:
            node.right = TreeNode(arr[i]); q.append(node.right)
        i += 1
    return root

examples = [
    (["FindElements", "find", "find"], [[[-1, None, -1]], [1], [2]], [None, False, True]),
    (["FindElements", "find", "find", "find"], [[[-1, -1, -1, -1, -1]], [1], [3], [5]], [None, True, True, False]),
    (["FindElements", "find", "find", "find", "find"], [[[-1, None, -1, -1, None, -1]], [2], [3], [4], [5]],
     [None, True, False, False, True]),
]
print("Đã định nghĩa TreeNode, build_tree và", len(examples), "ví dụ")

Đã định nghĩa TreeNode, build_tree và 3 ví dụ


## 3. Kiến thức nền

### 3.1 Cây nhị phân và mảng level-order

LeetCode mô tả cây bằng mảng **theo tầng** (level-order): phần tử đầu là gốc, sau đó lần lượt con trái/phải của từng nút **đang tồn tại**; `null` (`None`) nghĩa là vị trí đó trống. Hàm `to_level` dưới đây làm ngược lại để kiểm tra `build_tree`.

In [2]:
def to_level(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append(None); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out

for arr in ([-1, None, -1], [-1, -1, -1, -1, -1], [-1, None, -1, -1, None, -1]):
    print(arr, "->", to_level(build_tree(arr)))

[-1, None, -1] -> [-1, None, -1]
[-1, -1, -1, -1, -1] -> [-1, -1, -1, -1, -1]
[-1, None, -1, -1, None, -1] -> [-1, None, -1, -1, None, -1]


### 3.2 Đánh số kiểu heap

Với gốc `0`, con trái `2x+1`, con phải `2x+2`: đây đúng là chỉ số mảng của **heap nhị phân**. Ngược lại, cha của `y > 0` là `(y − 1) // 2`, và `y` là con trái nếu `y` lẻ, con phải nếu `y` chẵn.

Tầng `d` (gốc ở tầng 0) chứa các giá trị từ `2^d − 1` đến `2^(d+1) − 2`.

In [3]:
for y in [1, 2, 3, 4, 5, 6, 13, 14]:
    side = "trái" if y % 2 == 1 else "phải"
    print(f"y={y:2}: cha={(y - 1) // 2}, là con {side}, tầng {(y + 1).bit_length() - 1}")

y= 1: cha=0, là con trái, tầng 1
y= 2: cha=0, là con phải, tầng 1
y= 3: cha=1, là con trái, tầng 2
y= 4: cha=1, là con phải, tầng 2
y= 5: cha=2, là con trái, tầng 2
y= 6: cha=2, là con phải, tầng 2
y=13: cha=6, là con trái, tầng 3
y=14: cha=6, là con phải, tầng 3


### 3.3 DFS đệ quy để gán giá trị

DFS truyền giá trị xuống: `recover(node, x)` gán `node.val = x` rồi gọi `recover(node.left, 2x+1)`, `recover(node.right, 2x+2)`. Chiều cao ≤ 20 nên đệ quy **an toàn** (sâu tối đa ~21).

In [4]:
def recover(node, x, out):
    if node is None:
        return
    node.val = x
    out.append(x)
    recover(node.left, 2 * x + 1, out)
    recover(node.right, 2 * x + 2, out)

root = build_tree([-1, -1, -1, -1, -1])
vals = []
recover(root, 0, vals)
print("Thứ tự tiền thứ tự (preorder):", vals)
print("Cây sau khôi phục (level-order):", to_level(root))

Thứ tự tiền thứ tự (preorder): [0, 1, 3, 4, 2]
Cây sau khôi phục (level-order): [0, 1, 2, 3, 4]


### 3.4 `set` — tra cứu O(1)

Lưu mọi giá trị khôi phục vào `set` ⇒ `target in s` trung bình `O(1)`. So với tìm trong `list` (`O(n)`), đây là khác biệt lớn khi có 10⁴ truy vấn trên 10⁴ nút.

In [5]:
s = set(vals)
lst = list(vals)
print(3 in s, 5 in s, "| trong list:", 3 in lst, 5 in lst)

True False | trong list: True False


### 3.5 Bit của `target + 1` (liên hệ bài 1256)

Đánh số heap từ 1 (`y + 1`): gốc là `1`, con trái `2k`, con phải `2k+1`. Khi đó **biểu diễn nhị phân** của `target + 1` mô tả đường đi: bỏ bit `1` đầu, mỗi bit còn lại từ trái sang phải: `0` ⇒ rẽ trái, `1` ⇒ rẽ phải.

In [6]:
def path_of(target):
    bits = bin(target + 1)[3:]
    return ["L" if b == "0" else "R" for b in bits]

for t in [0, 1, 2, 3, 4, 5, 6, 13]:
    print(f"target={t:2}  target+1={bin(t + 1):>7}  đường đi={path_of(t)}")

target= 0  target+1=    0b1  đường đi=[]
target= 1  target+1=   0b10  đường đi=['L']
target= 2  target+1=   0b11  đường đi=['R']
target= 3  target+1=  0b100  đường đi=['L', 'L']
target= 4  target+1=  0b101  đường đi=['L', 'R']
target= 5  target+1=  0b110  đường đi=['R', 'L']
target= 6  target+1=  0b111  đường đi=['R', 'R']
target=13  target+1= 0b1110  đường đi=['R', 'R', 'L']


### 3.6 Bài thiết kế: chạy chuỗi lệnh kiểu LeetCode

LeetCode cho hai mảng: tên lệnh và tham số. Lệnh đầu là hàm tạo; các lệnh sau gọi phương thức trên đối tượng. Ta viết `run_commands(cls, ops, args)` trả về danh sách kết quả (`None` cho hàm tạo) — **đây là bản giả lập** để test.

In [7]:
def run_commands(cls, ops, args):
    obj, out = None, []
    for op, arg in zip(ops, args):
        if op == cls.__name__ or op == "FindElements":
            obj = cls(build_tree(arg[0]))
            out.append(None)
        else:
            out.append(getattr(obj, op)(*arg))
    return out

class Dummy:
    def __init__(self, root): self.root = root
    def find(self, target): return target == 0
print(run_commands(Dummy, *examples[0][:2]))

[None, False, False]


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 3**: mảng `[-1, null, -1, -1, null, -1]`.

```
        0
         \
          2            (con phải của 0: 2·0+2 = 2)
         /
        5              (con trái của 2: 2·2+1 = 5)
       /
     11                (con trái của 5: 2·5+1 = 11)
```

Đọc mảng theo tầng: phần tử 0 là gốc; phần tử 1–2 là (trái, phải) của gốc = (`null`, `-1`) ⇒ chỉ có con phải (giá trị 2). Phần tử 3–4 là (trái, phải) của nút 2 = (`-1`, `null`) ⇒ con trái (giá trị 5). Phần tử 5 là con **trái** của nút 5 (giá trị 11). Lưu ý: các vị trí `null` **không** sinh ra cặp con trong mảng.

Vậy tập giá trị = `{0, 2, 5, 11}` ⇒ `find(2)=True, find(3)=False, find(4)=False, find(5)=True` — khớp đề.

In [8]:
root = build_tree([-1, None, -1, -1, None, -1])
vals = []
recover(root, 0, vals)
print("Giá trị khôi phục:", sorted(vals))
print("Level-order:", to_level(root))
print([t in set(vals) for t in [2, 3, 4, 5]])

Giá trị khôi phục: [0, 2, 5, 11]
Level-order: [0, None, 2, 5, None, 11]
[True, False, False, True]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** khôi phục cây trong `__init__` (DFS), giữ lại `root`. Mỗi lần `find(target)`, **duyệt cả cây** (hoặc danh sách giá trị) tìm `target`.

Đúng nhưng mỗi truy vấn `O(n)`.

In [9]:
class FindElementsBrute:
    def __init__(self, root: Optional[TreeNode]):
        self.root = root
        def dfs(node, x):
            if node:
                node.val = x
                dfs(node.left, 2 * x + 1)
                dfs(node.right, 2 * x + 2)
        dfs(root, 0)

    def find(self, target: int) -> bool:
        stack = [self.root] if self.root else []
        while stack:
            node = stack.pop()
            if node.val == target:
                return True
            if node.left: stack.append(node.left)
            if node.right: stack.append(node.right)
        return False

for ops, args, want in examples:
    print(run_commands(FindElementsBrute, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


**Độ phức tạp:** `__init__` `O(n)`; mỗi `find` `O(n)` ⇒ với `q` truy vấn: **`O(n + q·n)`** — tới `10^8` thao tác Python khi `n = q = 10^4`, quá chậm. Bộ nhớ `O(h)` cho stack.

**Nút thắt:** tập giá trị **không đổi** sau khi khôi phục — tính một lần, lưu vào cấu trúc tra cứu nhanh.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — tiền xử lý:** dữ liệu tĩnh + nhiều truy vấn ⇒ trả chi phí một lần ở hàm tạo, lưu `set` giá trị ⇒ `find` `O(1)`.

**Insight 2 — giá trị mã hoá đường đi:** giá trị `x` của một nút **xác định duy nhất** đường từ gốc tới vị trí của nó (vì cha của `x` là `(x−1)//2`, và lẻ/chẵn cho biết trái/phải). Nên để kiểm tra `target` có tồn tại, chỉ cần **đi theo đường đó** từ gốc: nếu mọi nút trên đường đều tồn tại thì `True`.

**Quy tắc "nếu… thì…":**
- Nếu bit tiếp theo của `target+1` (sau bit đầu) là `0` ⇒ đi `left`; là `1` ⇒ đi `right`.
- Nếu gặp `None` giữa đường ⇒ `False`.
- Nếu `target + 1` có nhiều hơn `21` bit (sâu hơn chiều cao tối đa 20) ⇒ chắc chắn `False`.

**Invariant (đi theo bit):** sau khi đọc `k` bit, nút hiện tại (nếu khác `None`) có giá trị bằng `(tiền tố k+1 bit của target+1) − 1`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — DFS khôi phục + `set` (lời giải chuẩn)

1. `__init__`: DFS gán `val`, đồng thời thêm vào `self.seen`.
2. `find`: `return target in self.seen`.

In [10]:
class FindElementsSet:
    def __init__(self, root: Optional[TreeNode]):
        self.seen = set()
        def dfs(node, x):
            if node is None:
                return
            node.val = x
            self.seen.add(x)
            dfs(node.left, 2 * x + 1)
            dfs(node.right, 2 * x + 2)
        dfs(root, 0)

    def find(self, target: int) -> bool:
        return target in self.seen

for ops, args, want in examples:
    print(run_commands(FindElementsSet, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


**Độ phức tạp Cách 2:** `__init__` `O(n)` thời gian, `O(n)` bộ nhớ (set) + `O(h)` đệ quy; `find` `O(1)` trung bình.

### 7.2 Cách 3 — BFS khôi phục (không đệ quy) + `set`

Giống Cách 2 nhưng dùng hàng đợi — hữu ích nếu chiều cao không bị giới hạn (cây lệch 10⁴ tầng sẽ tràn đệ quy).

In [11]:
class FindElementsBFS:
    def __init__(self, root: Optional[TreeNode]):
        self.seen = set()
        if root is None:
            return
        root.val = 0
        q = deque([root])
        while q:
            node = q.popleft()
            x = node.val
            self.seen.add(x)
            if node.left:
                node.left.val = 2 * x + 1
                q.append(node.left)
            if node.right:
                node.right.val = 2 * x + 2
                q.append(node.right)

    def find(self, target: int) -> bool:
        return target in self.seen

for ops, args, want in examples:
    print(run_commands(FindElementsBFS, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


**Độ phức tạp Cách 3:** như Cách 2: `O(n)` khởi tạo, `O(1)` mỗi truy vấn, `O(n)` bộ nhớ (set + hàng đợi tối đa bằng độ rộng cây).

### 7.3 Cách 4 — Đi theo bit của `target + 1` (không cần `set`, không cần khôi phục)

1. `__init__`: chỉ lưu `root` (thậm chí không cần sửa `val`).
2. `find(target)`: `bits = bin(target + 1)[3:]`; từ gốc, với mỗi bit: `0` → `left`, `1` → `right`; gặp `None` ⇒ `False`. Đi hết ⇒ `True`.

Vì chiều cao ≤ 20, mỗi `find` tối đa ~20 bước; `target ≤ 10^6` thì `target+1` có ≤ 20 bit.

In [12]:
class FindElementsBits:
    def __init__(self, root: Optional[TreeNode]):
        self.root = root

    def find(self, target: int) -> bool:
        node = self.root
        for b in bin(target + 1)[3:]:
            if node is None:
                return False
            node = node.left if b == "0" else node.right
        return node is not None

for ops, args, want in examples:
    print(run_commands(FindElementsBits, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


Phiên bản dùng toán tử bit (không tạo xâu): duyệt bit từ vị trí `k−2` xuống `0`, với `k = (target+1).bit_length()`.

In [13]:
class FindElementsBits2:
    def __init__(self, root: Optional[TreeNode]):
        self.root = root

    def find(self, target: int) -> bool:
        x = target + 1
        node = self.root
        for i in range(x.bit_length() - 2, -1, -1):
            if node is None:
                return False
            node = node.right if (x >> i) & 1 else node.left
        return node is not None

for ops, args, want in examples:
    print(run_commands(FindElementsBits2, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


**Độ phức tạp Cách 4:** `__init__` `O(1)`; mỗi `find` `O(log target)` = `O(min(h, log target))` ≤ 20 bước; bộ nhớ phụ **`O(1)`**. Đánh đổi: chậm hơn `set` một hằng số nhỏ mỗi truy vấn, nhưng không tốn `O(n)` bộ nhớ và không phải duyệt cả cây lúc khởi tạo.

### 7.4 Cách 5 — Đi ngược từ `target` lên gốc (biến thể của Cách 4)

Tính dãy tổ tiên của `target` bằng `(y−1)//2` cho tới `0`, đảo ngược, rồi đi xuống từ gốc kiểm tra từng bước. Cùng ý tưởng, nhưng lập luận bằng công thức cha thay vì bit — dễ nhớ nếu đã quen heap.

In [14]:
class FindElementsParent:
    def __init__(self, root: Optional[TreeNode]):
        self.root = root

    def find(self, target: int) -> bool:
        steps = []
        y = target
        while y > 0:
            steps.append("L" if y % 2 == 1 else "R")
            y = (y - 1) // 2
        node = self.root
        for s in reversed(steps):
            if node is None:
                return False
            node = node.left if s == "L" else node.right
        return node is not None

for ops, args, want in examples:
    print(run_commands(FindElementsParent, ops, args), "| mong đợi", want)

[None, False, True] | mong đợi [None, False, True]
[None, True, True, False] | mong đợi [None, True, True, False]
[None, True, False, False, True] | mong đợi [None, True, False, False, True]


## 8. TRACE từng bước

**Trace Cách 4** với cây Ví dụ 3 (giá trị `{0, 2, 5, 11}`):

| `target` | `target+1` (nhị phân) | bit sau bit đầu | đường đi | kết quả |
|---|---|---|---|---|
| 2 | `11` | `1` | R → nút 2 | True |
| 3 | `100` | `00` | L → None | False |
| 5 | `110` | `10` | R (2) → L (5) | True |
| 11 | `1100` | `100` | R (2) → L (5) → L (11) | True |
| 6 | `111` | `11` | R (2) → R → None | False |

In [15]:
def trace_find_bits(root, target):
    node = root
    bits = bin(target + 1)[3:]
    print(f"find({target}): target+1 = {bin(target + 1)[2:]}, các bit đường đi = {bits!r}")
    cur = 0
    for b in bits:
        if node is None:
            print("   gặp None -> False")
            return False
        cur = 2 * cur + (1 if b == "0" else 2)
        node = node.left if b == "0" else node.right
        print(f"   bit {b} -> rẽ {'trái' if b == '0' else 'phải'}, giá trị kỳ vọng {cur:3}, nút {'tồn tại' if node else 'None'}")
    print("   =>", node is not None)
    return node is not None

root = build_tree([-1, None, -1, -1, None, -1])
for t in [2, 3, 5, 11, 6]:
    trace_find_bits(root, t)

find(2): target+1 = 11, các bit đường đi = '1'
   bit 1 -> rẽ phải, giá trị kỳ vọng   2, nút tồn tại
   => True
find(3): target+1 = 100, các bit đường đi = '00'
   bit 0 -> rẽ trái, giá trị kỳ vọng   1, nút None
   gặp None -> False
find(5): target+1 = 110, các bit đường đi = '10'
   bit 1 -> rẽ phải, giá trị kỳ vọng   2, nút tồn tại
   bit 0 -> rẽ trái, giá trị kỳ vọng   5, nút tồn tại
   => True
find(11): target+1 = 1100, các bit đường đi = '100'
   bit 1 -> rẽ phải, giá trị kỳ vọng   2, nút tồn tại
   bit 0 -> rẽ trái, giá trị kỳ vọng   5, nút tồn tại
   bit 0 -> rẽ trái, giá trị kỳ vọng  11, nút tồn tại
   => True
find(6): target+1 = 111, các bit đường đi = '11'
   bit 1 -> rẽ phải, giá trị kỳ vọng   2, nút tồn tại
   bit 1 -> rẽ phải, giá trị kỳ vọng   6, nút None
   => False


In [16]:
def trace_recover_bfs(arr):
    root = build_tree(arr)
    root.val = 0
    q = deque([root]); step = 0
    while q:
        node = q.popleft(); step += 1
        kids = []
        if node.left:
            node.left.val = 2 * node.val + 1; q.append(node.left); kids.append(("L", node.left.val))
        if node.right:
            node.right.val = 2 * node.val + 2; q.append(node.right); kids.append(("R", node.right.val))
        print(f"bước {step}: lấy nút {node.val:2}, gán con {kids}, hàng đợi = {[n.val for n in q]}")
    return root

trace_recover_bfs([-1, -1, -1, -1, -1]);

bước 1: lấy nút  0, gán con [('L', 1), ('R', 2)], hàng đợi = [1, 2]
bước 2: lấy nút  1, gán con [('L', 3), ('R', 4)], hàng đợi = [2, 3, 4]
bước 3: lấy nút  2, gán con [], hàng đợi = [3, 4]
bước 4: lấy nút  3, gán con [], hàng đợi = [4]
bước 5: lấy nút  4, gán con [], hàng đợi = []


## 9. Code nộp LeetCode

Dùng Cách 2 (DFS + set), đúng tên lớp/phương thức như skeleton. Trên LeetCode, `TreeNode` đã có sẵn nên annotation `TreeNode | None` hợp lệ (Python ≥ 3.10).

In [17]:
class FindElements:

    def __init__(self, root: TreeNode | None):
        self.seen = set()

        def dfs(node, x):
            if node is None:
                return
            node.val = x
            self.seen.add(x)
            dfs(node.left, 2 * x + 1)
            dfs(node.right, 2 * x + 2)

        dfs(root, 0)

    def find(self, target: int) -> bool:
        return target in self.seen


# Your FindElements object will be instantiated and called as such:
# obj = FindElements(root)
# param_1 = obj.find(target)
for ops, args, want in examples:
    got = run_commands(FindElements, ops, args)
    print(got, "OK" if got == want else "SAI")

[None, False, True] OK
[None, True, True, False] OK
[None, True, False, False, True] OK


## 10. Vì sao thuật toán đúng

**Cách 2/3.** Bằng quy nạp theo độ sâu: gốc nhận `0` (đúng quy tắc). Nếu một nút được gán đúng giá trị `x` thì DFS/BFS gán cho con trái `2x+1`, con phải `2x+2` — đúng định nghĩa. Mọi nút đều được thăm (DFS/BFS duyệt hết cây) ⇒ `seen` = đúng tập giá trị của cây đã khôi phục ⇒ `find` đúng.

**Cách 4.** Gọi `v(node)` là giá trị khôi phục, `w = v + 1`. Gốc có `w = 1`; con trái có `w' = 2(v+1) = 2w`, con phải `w' = 2v + 3 = 2w + 1`. Nghĩa là đi xuống trái = **thêm bit `0`** vào cuối nhị phân của `w`, xuống phải = **thêm bit `1`**. Do đó nút có giá trị `target` (nếu tồn tại) nằm ở cuối đường đi mô tả bởi các bit của `target + 1` sau bit đầu, và đường đó **duy nhất**. `target` tồn tại ⇔ mọi nút trên đường đó tồn tại. ∎

(Đây chính là ánh xạ của bài 1256 "Encode Number": `bin(target+1)[3:]` là "mã" đường đi.)

## 11. Phân tích Time / Space

Gọi `n` = số nút (≤ 10⁴), `h` = chiều cao (≤ 20), `q` = số lần gọi `find` (≤ 10⁴), `T` = `target` (≤ 10⁶).

| Cách | `__init__` | `find` | Bộ nhớ phụ | Tổng cho `q` truy vấn |
|---|---|---|---|---|
| 1. Brute | `O(n)` | `O(n)` | `O(h)` | `O(n + q·n)` |
| 2. DFS + set | `O(n)` | `O(1)` TB | `O(n)` | `O(n + q)` |
| 3. BFS + set | `O(n)` | `O(1)` TB | `O(n)` | `O(n + q)` |
| 4. Theo bit | `O(1)` | `O(min(h, log T))` | `O(1)` | `O(q · log T)` |
| 5. Ngược lên gốc | `O(1)` | `O(log T)` | `O(log T)` | `O(q · log T)` |

`set` có worst case `O(n)` mỗi tra cứu (va chạm băm), nhưng với số nguyên nhỏ thì thực tế là `O(1)`. Cách 4 có cận **tất định** ≤ 20 bước.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [18]:
ALL = [FindElementsBrute, FindElementsSet, FindElementsBFS, FindElementsBits,
       FindElementsBits2, FindElementsParent, FindElements]
edge = [
    (["FindElements", "find", "find"], [[[-1]], [0], [1]], [None, True, False]),          # chỉ có gốc
    (["FindElements", "find", "find", "find"], [[[-1, -1]], [1], [2], [0]], [None, True, False, True]),
    (["FindElements", "find"], [[[-1, None, -1]], [10**6]], [None, False]),             # target lớn
    (["FindElements", "find", "find"], [[[-1, -1, -1, None, -1]], [4], [3]], [None, True, False]),
]
for cls in ALL:
    for ops, args, want in examples + edge:
        got = run_commands(cls, ops, args)
        assert got == want, (cls.__name__, ops, args, got, want)
print("Mọi lớp đúng với", len(examples + edge), "chuỗi lệnh")

Mọi lớp đúng với 7 chuỗi lệnh


### 12.2 Cây ngẫu nhiên — đối chiếu với brute force (seed cố định)

Sinh cây ngẫu nhiên chiều cao ≤ 20 với tối đa vài trăm nút; truy vấn cả giá trị có trong cây lẫn giá trị ngẫu nhiên trong `[0, 10^6]`.

In [19]:
def random_tree_arr(rng, max_nodes, max_h=20, p=0.6):
    """Sinh cây ngẫu nhiên rồi xuất ra mảng level-order toàn -1/None."""
    root = TreeNode(-1); count = 1
    frontier = [(root, 0)]
    while frontier and count < max_nodes:
        node, d = frontier.pop(rng.randrange(len(frontier)))
        if d >= max_h:
            continue
        for side in ("left", "right"):
            if count < max_nodes and rng.random() < p:
                child = TreeNode(-1); setattr(node, side, child); count += 1
                frontier.append((child, d + 1))
    return to_level(root)

def collect_values(arr):
    root = build_tree(arr); out = []
    recover(root, 0, out)
    return out

rng = random.Random(1261)
for _ in range(200):
    arr = random_tree_arr(rng, rng.randint(1, 300), p=rng.choice([0.4, 0.6, 0.9]))
    vals = collect_values(arr)
    queries = [rng.choice(vals) for _ in range(20)] + [rng.randint(0, 10**6) for _ in range(20)] + list(range(30))
    ops = ["FindElements"] + ["find"] * len(queries)
    args = [[arr]] + [[t] for t in queries]
    want = [None] + [t in set(vals) for t in queries]
    for cls in ALL:
        assert run_commands(cls, ops, args) == want, cls.__name__
print("200 cây ngẫu nhiên × 7 lớp: tất cả khớp")

200 cây ngẫu nhiên × 7 lớp: tất cả khớp


### 12.3 Giới hạn: 10⁴ nút, 10⁴ truy vấn

In [20]:
arr = random_tree_arr(random.Random(7), 10**4, p=0.95)
vals = collect_values(arr)
print("Số nút:", len(vals), "| chiều cao ≤ 20:", max(v + 1 for v in vals).bit_length() - 1 <= 20)
queries = [random.Random(8).randint(0, 10**6) for _ in range(10**4)]
ops = ["FindElements"] + ["find"] * len(queries)
args = [[arr]] + [[t] for t in queries]
ref = None
for cls in [FindElementsSet, FindElementsBFS, FindElementsBits, FindElementsBits2, FindElements, FindElementsBrute]:
    a = args if cls is not FindElementsBrute else args[:301]
    t0 = time.perf_counter()
    res = run_commands(cls, ops[:len(a)], a)
    dt = time.perf_counter() - t0
    if ref is None:
        ref = res
    assert res == ref[:len(res)]
    print(f"{cls.__name__:20}: {len(a) - 1:5} truy vấn trong {dt:.3f}s")

Số nút: 10000 | chiều cao ≤ 20: True


FindElementsSet     : 10000 truy vấn trong 0.009s
FindElementsBFS     : 10000 truy vấn trong 0.007s
FindElementsBits    : 10000 truy vấn trong 0.009s
FindElementsBits2   : 10000 truy vấn trong 0.015s
FindElements        : 10000 truy vấn trong 0.007s


FindElementsBrute   :   300 truy vấn trong 0.256s


Brute force chỉ chạy 300 truy vấn mà đã tốn thời gian tương đương 10⁴ truy vấn của các cách khác — đúng như phân tích `O(q·n)`.

## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nhầm công thức con (đánh số từ 1: `2x`, `2x+1`)

Quen với heap đánh số từ 1 nên viết con trái `2x`, con phải `2x+1` — với gốc `0` thì con trái cũng là `0`!

In [21]:
class FindElementsWrong1:
    def __init__(self, root):
        self.seen = set()
        def dfs(node, x):
            if node:
                self.seen.add(x)
                dfs(node.left, 2 * x)          # SAI
                dfs(node.right, 2 * x + 1)     # SAI
        dfs(root, 0)
    def find(self, target):
        return target in self.seen

ops, args, want = examples[1]
print("Sai :", run_commands(FindElementsWrong1, ops, args))
print("Đúng:", want)

Sai : [None, True, False, False]
Đúng: [None, True, True, False]


### Lỗi 2 — Đi theo bit của `target` thay vì `target + 1`

Quên dịch sang đánh số từ 1 ⇒ đường đi sai hoàn toàn.

In [22]:
class FindElementsWrong2:
    def __init__(self, root):
        self.root = root
    def find(self, target):
        node = self.root
        for b in bin(target)[3:]:              # SAI: thiếu + 1
            if node is None:
                return False
            node = node.left if b == "0" else node.right
        return node is not None

for ops, args, want in examples:
    print("Sai :", run_commands(FindElementsWrong2, ops, args), "| Đúng:", want)

Sai : [None, True, False] | Đúng: [None, False, True]
Sai : [None, True, True, True] | Đúng: [None, True, True, False]
Sai : [None, False, True, False, False] | Đúng: [None, True, False, False, True]


### Lỗi 3 — Đọc bit từ thấp lên cao (sai chiều)

Lấy bit bằng `x & 1; x >>= 1` rồi đi xuống ngay — đó là đường đi **từ dưới lên**, phải đảo ngược trước khi đi từ gốc xuống.

In [23]:
class FindElementsWrong3:
    def __init__(self, root):
        self.root = root
    def find(self, target):
        x = target + 1
        node = self.root
        while x > 1:                           # SAI: bit thấp trước
            if node is None:
                return False
            node = node.right if x & 1 else node.left
            x >>= 1
        return node is not None

ops = ["FindElements", "find", "find"]
args = [[[-1, None, -1, -1, None, -1]], [5], [4]]
print("Sai :", run_commands(FindElementsWrong3, ops, args))
print("Đúng:", run_commands(FindElements, ops, args))

Sai : [None, False, True]
Đúng: [None, True, False]


Với `5`: `5+1 = 110₂`, đúng là R rồi L; đọc ngược thành L rồi R ⇒ gặp `None` ở bước đầu (gốc không có con trái) ⇒ `False` sai. Còn `4` (`101₂`: L, R) đọc ngược thành R, L ⇒ tới nút 5 ⇒ `True` sai!

### Lỗi 4 — Giả định cây đầy đủ: `find(t)` ⇔ `t < số nút`

Đếm số nút `n` rồi trả `target < n`. Chỉ đúng với cây **đầy đủ** (complete), sai với cây có lỗ.

In [24]:
class FindElementsWrong4:
    def __init__(self, root):
        def count(node):
            return 0 if node is None else 1 + count(node.left) + count(node.right)
        self.n = count(root)
    def find(self, target):
        return target < self.n                  # SAI với cây không đầy đủ

for ops, args, want in examples:
    print("Sai :", run_commands(FindElementsWrong4, ops, args), "| Đúng:", want)

Sai : [None, True, False] | Đúng: [None, False, True]
Sai : [None, True, True, False] | Đúng: [None, True, True, False]
Sai : [None, True, True, False, False] | Đúng: [None, True, False, False, True]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"gốc 0, con trái 2x+1, con phải 2x+2"** ⇒ heap indexing; nghĩ ngay tới `bin(x+1)` để mã hoá đường đi.
- **Bài Design "khởi tạo một lần + truy vấn nhiều lần"** ⇒ dồn việc nặng vào `__init__`, truy vấn dùng `set`/`dict`/mảng đã tính sẵn.
- Ràng buộc "chiều cao ≤ 20" là gợi ý: đệ quy an toàn, và đường đi ≤ 20 bước ⇒ Cách 4 rất nhanh.
- Viết sẵn helper `build_tree(arr)` + `run_commands(...)` để tự test bài thiết kế — tiết kiệm rất nhiều thời gian debug.
- Khi `target` có thể vượt mọi giá trị trong cây, Cách 4 tự xử lý (gặp `None` sớm) — không cần kiểm tra thêm.

## 15. Biến thể của bài

1. **Thêm nút sau khi khởi tạo** (`add(parentVal, isLeft)`): với Cách 2 chỉ cần thêm `2x+1`/`2x+2` vào set.
2. **Đếm số giá trị trong đoạn `[lo, hi]`**: lưu danh sách giá trị đã sắp xếp + `bisect`.
3. **LeetCode 662 — Maximum Width of Binary Tree**: dùng chính cách đánh số heap để tính độ rộng mỗi tầng.
4. **LeetCode 1104 — Path In Zigzag Labelled Binary Tree**: đánh số heap nhưng đảo chiều ở tầng chẵn/lẻ.
5. **LeetCode 222 — Count Complete Tree Nodes**: kiểm tra sự tồn tại của nút cuối theo bit, `O(log² n)`.

Minh hoạ biến thể 2 (đếm trong đoạn) và 3 (độ rộng lớn nhất):

In [25]:
import bisect

class FindElementsRange(FindElementsSet):
    def __init__(self, root):
        super().__init__(root)
        self.sorted_vals = sorted(self.seen)
    def count_range(self, lo, hi):
        return bisect.bisect_right(self.sorted_vals, hi) - bisect.bisect_left(self.sorted_vals, lo)

def max_width(arr):
    vals = collect_values(arr)
    by_level = {}
    for v in vals:
        d = (v + 1).bit_length() - 1
        lo, hi = by_level.get(d, (v, v))
        by_level[d] = (min(lo, v), max(hi, v))
    return max(hi - lo + 1 for lo, hi in by_level.values())

fe = FindElementsRange(build_tree([-1, -1, -1, -1, -1]))
print("Số giá trị trong [1, 3]:", fe.count_range(1, 3))
print("Độ rộng lớn nhất của [-1,-1,-1,-1,None,None,-1]:", max_width([-1, -1, -1, -1, None, None, -1]))

Số giá trị trong [1, 3]: 3
Độ rộng lớn nhất của [-1,-1,-1,-1,None,None,-1]: 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time (`init` / `find`) | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute | Khôi phục, mỗi `find` duyệt cả cây | `O(n)` / `O(n)` | `O(h)` | Chỉ để kiểm chứng |
| 2. DFS + set | Khôi phục bằng DFS, lưu set | `O(n)` / `O(1)` | `O(n)` | Lời giải chuẩn, ngắn |
| 3. BFS + set | Như 2 nhưng không đệ quy | `O(n)` / `O(1)` | `O(n)` | Cây có thể rất sâu |
| 4. Theo bit `target+1` | Đường đi được mã hoá trong bit | `O(1)` / `O(log T)` | `O(1)` | Bộ nhớ chặt, ít truy vấn, cây lớn |
| 5. Ngược lên gốc | Dãy cha `(y−1)//2` rồi đi xuống | `O(1)` / `O(log T)` | `O(log T)` | Khi quen công thức heap hơn bit |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Cây `[-1, -1, None, -1, -1]`. Chuỗi `find(3), find(4), find(2), find(7)` cho kết quả gì?

### Bài 2 (tính tay)
Viết đường đi (L/R) từ gốc tới nút có giá trị `20` và `29`. Mỗi nút nằm ở tầng mấy?

### Bài 3 (điền TODO)
Hoàn thiện `path_to(target)`: trả về chuỗi gồm `'L'`/`'R'` mô tả đường từ gốc tới vị trí `target`.

### Bài 4 (tự viết lại)
Viết lại lớp `FindElements` theo Cách 4 (bit) nhưng **có cache**: lưu kết quả các `target` đã hỏi trong `dict` — `FindElementsCached`.

### Bài 5 (mini challenge)
Viết `lca_value(a, b)`: giá trị của tổ tiên chung thấp nhất của hai nút có giá trị `a`, `b` trong cây heap vô hạn (không cần cây thật).

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def path_to(target):
    # TODO: dùng bin(target + 1)[3:], đổi '0' -> 'L', '1' -> 'R'
    pass

# Bài 4 — TODO
class FindElementsCached:
    def __init__(self, root):
        self.root = root
        self.cache = {}
    def find(self, target):
        # TODO: nếu target trong cache thì trả luôn; ngược lại đi theo bit rồi lưu
        pass

# Bài 5 — TODO
def lca_value(a, b):
    # TODO: lặp: trong khi a != b, đưa số lớn hơn lên cha (x - 1) // 2
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Dựng cây: gốc 0 có con trái 1 (không có con phải). (2) Nút 1 có con trái 3 và con phải 4. (3) `[True, True, False, False]`.

**Bài 2.** (1) `21 = 10101₂` ⇒ bỏ bit đầu `0101` ⇒ L R L R. (2) `30 = 11110₂` ⇒ `1110` ⇒ R R R L. (3) Cả hai ở tầng 4 (số bit − 1).

**Bài 3.** (1) `bin(target+1)[3:]`. (2) `str.translate` hoặc list comprehension. (3) `target = 0` ⇒ xâu rỗng.

**Bài 4.** (1) Tái dùng thân hàm của `FindElementsBits.find`. (2) `self.cache[target] = res`. (3) Kiểm tra bằng test ngẫu nhiên so với `FindElements`.

**Bài 5.** (1) Số lớn hơn không thể ở tầng nông hơn số nhỏ hơn. (2) Đưa số lớn hơn lên cha, lặp lại. (3) Tương đương: tiền tố chung dài nhất của `bin(a+1)` và `bin(b+1)`, trừ 1.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
ops = ["FindElements", "find", "find", "find", "find"]
args = [[[-1, -1, None, -1, -1]], [3], [4], [2], [7]]
assert run_commands(FindElements, ops, args) == [None, True, True, False, False]

# Bài 3
def path_to_sol(target):
    return "".join("L" if b == "0" else "R" for b in bin(target + 1)[3:])

# Bài 2
assert path_to_sol(20) == "LRLR" and path_to_sol(29) == "RRRL"

# Bài 4
class FindElementsCachedSol:
    def __init__(self, root):
        self.root = root
        self.cache = {}
    def find(self, target):
        if target in self.cache:
            return self.cache[target]
        node = self.root
        for b in bin(target + 1)[3:]:
            if node is None:
                break
            node = node.left if b == "0" else node.right
        self.cache[target] = node is not None
        return self.cache[target]

# Bài 5
def lca_value_sol(a, b):
    while a != b:
        if a > b:
            a = (a - 1) // 2
        else:
            b = (b - 1) // 2
    return a

def lca_prefix(a, b):
    x, y = bin(a + 1)[2:], bin(b + 1)[2:]
    k = 0
    while k < min(len(x), len(y)) and x[k] == y[k]:
        k += 1
    return int(x[:k], 2) - 1

assert path_to_sol(0) == ""
rng = random.Random(55)
for _ in range(100):
    arr = random_tree_arr(rng, rng.randint(1, 200))
    queries = [rng.randint(0, 600) for _ in range(30)]
    ops = ["FindElements"] + ["find"] * 60
    args = [[arr]] + [[t] for t in queries + queries]           # hỏi lặp để chạm cache
    assert run_commands(FindElementsCachedSol, ops, args) == run_commands(FindElements, ops, args)
for _ in range(2000):
    a, b = rng.randint(0, 10**6), rng.randint(0, 10**6)
    assert lca_value_sol(a, b) == lca_prefix(a, b)
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "khôi phục cây heap + set":**

```python
class FindElements:
    def __init__(self, root):
        self.seen = set()
        def dfs(node, x):
            if node:
                node.val = x
                self.seen.add(x)
                dfs(node.left, 2*x + 1)
                dfs(node.right, 2*x + 2)
        dfs(root, 0)
    def find(self, target):
        return target in self.seen
```

**Template "đi theo bit":**

```python
node = root
for b in bin(target + 1)[3:]:
    if node is None: return False
    node = node.left if b == '0' else node.right
return node is not None
```

**Checklist trước khi submit:**
- [ ] Công thức con: **`2x+1`** (trái), **`2x+2`** (phải), gốc `0`.
- [ ] DFS kiểm tra `node is None` trước khi truy cập `node.val`.
- [ ] Theo bit: dùng `target + 1`, cắt `[3:]`, đọc bit **từ trái sang phải**.
- [ ] Không giả định cây đầy đủ.
- [ ] Tự test bằng bộ chạy chuỗi lệnh (giả lập) với cả 3 ví dụ.
- [ ] Đúng tên lớp `FindElements`, phương thức `__init__(self, root)` và `find(self, target) -> bool`.